# Import libs

In [22]:
import sys
sys.path.append("../../../")

In [3]:
# statement_name = "/Users/apenkin/workspace/personal/finman/data/Справка_о_движении_денежных_средств_241114.pdf"
# acc_name = "t-bank (main)"

statement_name = "2025-06-2025-08_Справка_о_движении_денежных_средств (1).pdf"
# acc_name = "t-bank (main)"
# acc_name = "t-bank (proxy deposit)"

# statement_name = "../data/ozonbank_document_7093904.pdf"
# bank_name = "ozon"


# statement_name = "/Users/apenkin/workspace/personal/finman/data/Выписка_по_счёту_дебетовой_карты_241114.pdf"
# bank_name = "sber"

# statement_name = "../data/ya_241115.pdf"
# bank_name = "ya"

# Настройка

In [4]:
from src.finman.utils.pdf import extract_text_from_pdf
from src.finman.expenses import tbank, ozon, sber, yapay

In [5]:
trans_col = "Сумма операции в валюте карты"
date_col = "Дата операции"
time_col = "Время операции"

# Прочитать выписку

In [6]:
text = extract_text_from_pdf(statement_name)

In [7]:
text

'Исх. № 398a486e\nАКЦИОНЕРНОЕ ОБЩЕСТВО «ТБАНК»\nРОССИЯ, 127287, МОСКВА, УЛ. 2-Я ХУТОРСКАЯ, Д. 38А, СТР. 26\nТЕЛ.: +7 495 648-10-00,  TBANK.RU\nСправка о движении средств\n17.09.2025\nПенкин Артем Александрович\nАдрес места жительства: 108809, Г Москва, Д Постниково, Ул Космическая , д. 11\nО продукте\nДата заключения договора:  31.08.2018\nНомер договора:  5086962189\nНомер лицевого счета:  40817810200006235175\nДвижение средств за период с 01.06.2025 по 31.08.2025\nДата и время\nоперации\nДата\nсписания\nСумма в валюте\nоперации\nСумма операции\nв валюте карты\nОписание\nоперации\nНомер\nкарты\n31.08.2025\n15:36\n31.08.2025\n15:38\n-2 360.00 ₽\n-2 360.00 ₽\nОплата в FARSH Ф-9_P_QR\n5037\n31.08.2025\n07:23\n31.08.2025\n07:36\n-1 500.00 ₽\n-1 500.00 ₽\nОплата в OTO*boosty\nChelyabinsk RUS\n2475\n30.08.2025\n17:10\n30.08.2025\n17:11\n-1 000.00 ₽\n-1 000.00 ₽\nОплата услуг mBank.MTS\n5037\n30.08.2025\n10:18\n30.08.2025\n10:33\n-220.00 ₽\n-220.00 ₽\nОплата в VKUSVILL Moskva\nRUS\n2475\n30.

In [8]:
transactions_df = tbank.parse_operations(text)

In [9]:
transactions_df.head()

,id,Дата операции,Время операции,Сумма операции в валюте карты,Валюта карты,Сумма в валюте операции,Валюта операции,Описание операции,Номер карты
0,2b4cf1defb,2025-08-31,2025-08-31 15:36:00,-2360.0,₽,-2360.0,₽,Оплата в FARSH Ф-9_P_QR,5037
1,f94e0215dd,2025-08-31,2025-08-31 07:23:00,-1500.0,₽,-1500.0,₽,Оплата в OTO*boosty Chelyabinsk RUS,2475
2,41c4e656f3,2025-08-30,2025-08-30 17:10:00,-1000.0,₽,-1000.0,₽,Оплата услуг mBank.MTS,5037
3,2bbc54e31a,2025-08-30,2025-08-30 10:18:00,-220.0,₽,-220.0,₽,Оплата в VKUSVILL Moskva RUS,2475
4,8de3b473f6,2025-08-30,2025-08-30 10:01:00,-720.0,₽,-720.0,₽,Оплата в VKUSVILL Moskva RUS,2475


In [10]:
incomes, expences = tbank.extract_total_operations(text)
incomes, expences

(9406189.49, 9787975.19)

In [11]:
transactions_df.head()

,id,Дата операции,Время операции,Сумма операции в валюте карты,Валюта карты,Сумма в валюте операции,Валюта операции,Описание операции,Номер карты
0,2b4cf1defb,2025-08-31,2025-08-31 15:36:00,-2360.0,₽,-2360.0,₽,Оплата в FARSH Ф-9_P_QR,5037
1,f94e0215dd,2025-08-31,2025-08-31 07:23:00,-1500.0,₽,-1500.0,₽,Оплата в OTO*boosty Chelyabinsk RUS,2475
2,41c4e656f3,2025-08-30,2025-08-30 17:10:00,-1000.0,₽,-1000.0,₽,Оплата услуг mBank.MTS,5037
3,2bbc54e31a,2025-08-30,2025-08-30 10:18:00,-220.0,₽,-220.0,₽,Оплата в VKUSVILL Moskva RUS,2475
4,8de3b473f6,2025-08-30,2025-08-30 10:01:00,-720.0,₽,-720.0,₽,Оплата в VKUSVILL Moskva RUS,2475


In [12]:
import numpy as np
assert np.isclose(transactions_df.loc[transactions_df[trans_col] > 0][trans_col].sum(), incomes, atol=0.01)
assert np.isclose(transactions_df.loc[transactions_df[trans_col] < 0][trans_col].sum(), -expences, atol=0.01)

In [16]:
transactions_df[time_col].dtype

dtype('<M8[ns]')

# Split months

In [17]:
aug_txn_df = transactions_df.loc[transactions_df[time_col].dt.strftime("%Y-%m") == "2025-08"]
aug_txn_df.head()

,id,Дата операции,Время операции,Сумма операции в валюте карты,Валюта карты,Сумма в валюте операции,Валюта операции,Описание операции,Номер карты
0,2b4cf1defb,2025-08-31,2025-08-31 15:36:00,-2360.0,₽,-2360.0,₽,Оплата в FARSH Ф-9_P_QR,5037
1,f94e0215dd,2025-08-31,2025-08-31 07:23:00,-1500.0,₽,-1500.0,₽,Оплата в OTO*boosty Chelyabinsk RUS,2475
2,41c4e656f3,2025-08-30,2025-08-30 17:10:00,-1000.0,₽,-1000.0,₽,Оплата услуг mBank.MTS,5037
3,2bbc54e31a,2025-08-30,2025-08-30 10:18:00,-220.0,₽,-220.0,₽,Оплата в VKUSVILL Moskva RUS,2475
4,8de3b473f6,2025-08-30,2025-08-30 10:01:00,-720.0,₽,-720.0,₽,Оплата в VKUSVILL Moskva RUS,2475


In [18]:
aug_txn_df['Описание операции'].value_counts()

Описание операции
Оплата в FLOOR TEA MOSCOW RUS                  11
Оплата в VKUSVILL                              10
Внутренний перевод на договор                   9
Оплата в Vkusvill_5648_SBP                      8
Внутрибанковский перевод с договора             7
                                               ..
Закрытие вклада Т-Банк                          1
Оплата в OSOBENNYI BARBER Moskva RUS            1
Оплата в _PVP 679 KM M11 NEVA_ Shushary RUS     1
Оплата в WHSD SOUTH SANKT-PETERBU RUS           1
Оплата в B131*boosty Chelyabinsk RUS            1
Name: count, Length: 128, dtype: int64

In [19]:
import pandas as pd

# Ваши транзакции (пример)
df = pd.DataFrame([
    {"date": "2025-09-01", "amount": -1200, "merchant": "Пятёрочка"},
    {"date": "2025-09-02", "amount": -550,  "merchant": "Кофейня №7"},
    {"date": "2025-09-03", "amount": -399,  "merchant": "МТС"},
    {"date": "2025-09-04", "amount": -3400, "merchant": "Ozon"},
])

# Если в df уже есть колонки 'category'/'subcategory', они будут подхвачены
for col in ["category", "subcategory"]:
    if col not in df.columns:
        df[col] = None

# Справочник: верхний уровень -> подуровни
CATEGORY_TREE = {
    "Еда": ["Продукты", "Кафе/Рестораны", "Доставка"],
    "Транспорт": ["Общественный", "Такси", "Личный авто"],
    "Связь": ["Мобильная связь", "Интернет", "Прочее"],
    "Покупки": ["Онлайн", "Оффлайн", "Подписки"],
    "Другое": ["Подарки", "Здоровье", "Прочее"],
}
TOP_LEVEL = list(CATEGORY_TREE.keys())

In [23]:
import ipywidgets as W
from IPython.display import display, clear_output

rows_ui = []
row_states = []  # сюда будем складывать состояние для каждой строки

def make_row_widget(idx, record):
    # Начальные значения (если уже размечали раньше)
    init_cat = record.get("category") if pd.notna(record.get("category")) else None
    init_sub = record.get("subcategory") if pd.notna(record.get("subcategory")) else None

    dd_cat = W.Dropdown(
        options=["—"] + TOP_LEVEL,
        value=init_cat if init_cat in TOP_LEVEL else "—",
        layout=W.Layout(width="220px")
    )
    dd_sub = W.Dropdown(
        options=["—"],
        value="—",
        layout=W.Layout(width="240px")
    )

    # Если есть выбранная категория, подставим её подкатегории
    def set_sub_options(cat):
        opts = ["—"]
        if cat in CATEGORY_TREE:
            opts += CATEGORY_TREE[cat]
        # сохранить текущее значение, если оно валидно
        current = dd_sub.value
        dd_sub.options = opts
        dd_sub.value = current if current in opts else "—"

    set_sub_options(dd_cat.value)

    # Когда меняется верхний — обновляем низ
    def on_cat_change(change):
        if change["name"] == "value":
            set_sub_options(change["new"])
    dd_cat.observe(on_cat_change, names="value")

    # Немного инфо по строке
    left = W.HBox([
        W.Label(f"#{idx+1}", layout=W.Layout(width="40px")),
        W.Label(str(record["date"]), layout=W.Layout(width="110px")),
        W.Label(f'{record["amount"]:+}', layout=W.Layout(width="80px")),
        W.Label(record["merchant"], layout=W.Layout(width="200px")),
    ])
    right = W.HBox([dd_cat, dd_sub])

    # Храним ссылки на виджеты и индекс строки
    state = {"idx": idx, "dd_cat": dd_cat, "dd_sub": dd_sub}
    row_states.append(state)

    # Обрамим в карточку
    box = W.HBox([left, right])
    box.layout = W.Layout(justify_content="space-between", border="1px solid #ddd", padding="6px", margin="3px 0")
    return box

# Заголовок (как шапка таблицы)
header = W.HBox([
    W.Label("", layout=W.Layout(width="40px")),
    W.Label("Дата", layout=W.Layout(width="110px")),
    W.Label("Сумма", layout=W.Layout(width="80px")),
    W.Label("Мерчант", layout=W.Layout(width="200px")),
    W.Label("Категория", layout=W.Layout(width="220px", margin="0 0 0 60px")),
    W.Label("Подкатегория", layout=W.Layout(width="240px")),
])
header.layout = W.Layout(border="0", padding="2px 6px", margin="0 0 6px 0")

rows_ui = [make_row_widget(i, df.iloc[i].to_dict()) for i in range(len(df))]
rows_box = W.VBox([header] + rows_ui)

# Кнопки действий
out = W.Output()

def save_to_df(_):
    with out:
        clear_output()
        for st in row_states:
            i = st["idx"]
            cat = st["dd_cat"].value if st["dd_cat"].value != "—" else None
            sub = st["dd_sub"].value if st["dd_sub"].value != "—" else None
            df.at[i, "category"] = cat
            df.at[i, "subcategory"] = sub
        display(df)

def export_csv(_):
    # путь можно поменять
    path = "labeled_transactions.csv"
    df.to_csv(path, index=False)
    with out:
        clear_output()
        print(f"Экспортировано в {path}")

btn_save = W.Button(description="Сохранить в DataFrame")
btn_save.on_click(save_to_df)

btn_export = W.Button(description="Экспорт в CSV")
btn_export.on_click(export_csv)

toolbar = W.HBox([btn_save, btn_export])
ui = W.VBox([rows_box, toolbar, out])
display(ui)

In [24]:
# --- v2: Разметка транзакций с фильтрами и "протяжкой" ---
import re
import pandas as pd
import ipywidgets as W
from IPython.display import display, clear_output

# ========= 1) Демо-данные (замените на свои) =========
df = pd.DataFrame([
    {"date": "2025-09-01", "amount": -1200, "merchant": "Пятёрочка"},
    {"date": "2025-09-02", "amount": -550,  "merchant": "Кофейня №7"},
    {"date": "2025-09-03", "amount": -399,  "merchant": "МТС"},
    {"date": "2025-09-04", "amount": -3400, "merchant": "Ozon"},
    {"date": "2025-09-05", "amount": -980,  "merchant": "Пятёрочка"},
])

for col in ["category", "subcategory"]:
    if col not in df.columns:
        df[col] = None

# ========= 2) Справочник категорий =========
CATEGORY_TREE = {
    "Еда": ["Продукты", "Кафе/Рестораны", "Доставка", "Супермаркет"],
    "Транспорт": ["Общественный", "Такси", "Личный авто"],
    "Связь": ["Мобильная связь", "Интернет", "Прочее"],
    "Покупки": ["Онлайн", "Оффлайн", "Подписки"],
    "Другое": ["Подарки", "Здоровье", "Прочее"],
}
TOP_LEVEL = list(CATEGORY_TREE.keys())

# ========= 3) Вспомогательные штуки =========
row_states = []           # видимые строки (обновляются при фильтрации)
visible_indices = []      # индексы df, которые сейчас показаны
out = W.Output()

def _mk_dd_cat(value=None, width="220px"):
    dd = W.Dropdown(options=["—"] + TOP_LEVEL,
                    value=(value if value in TOP_LEVEL else "—"),
                    layout=W.Layout(width=width))
    return dd

def _mk_dd_sub(cat_value, init_sub=None, width="240px"):
    opts = ["—"] + (CATEGORY_TREE.get(cat_value, []) if cat_value in CATEGORY_TREE else [])
    sub_val = init_sub if init_sub in opts else "—"
    dd = W.Dropdown(options=opts, value=sub_val, layout=W.Layout(width=width))
    return dd

def _set_sub_options(dd_sub, cat_value):
    opts = ["—"] + (CATEGORY_TREE.get(cat_value, []) if cat_value in CATEGORY_TREE else [])
    current = dd_sub.value
    dd_sub.options = opts
    dd_sub.value = current if current in opts else "—"

def make_row_widget(idx, record):
    dd_cat = _mk_dd_cat(record.get("category"))
    dd_sub = _mk_dd_sub(dd_cat.value, record.get("subcategory"))

    def on_cat_change(ch):
        if ch["name"] == "value":
            _set_sub_options(dd_sub, ch["new"])
    dd_cat.observe(on_cat_change, names="value")

    left = W.HBox([
        W.Label(f"#{idx+1}", layout=W.Layout(width="40px")),
        W.Label(str(record["date"]), layout=W.Layout(width="110px")),
        W.Label(f'{record["amount"]:+}', layout=W.Layout(width="80px")),
        W.Label(str(record["merchant"]), layout=W.Layout(width="220px")),
    ])
    right = W.HBox([dd_cat, dd_sub])

    state = {"idx": idx, "dd_cat": dd_cat, "dd_sub": dd_sub}
    box = W.HBox([left, right])
    box.layout = W.Layout(justify_content="space-between", border="1px solid #ddd",
                          padding="6px", margin="3px 0")
    return box, state

# ========= 4) Фильтры =========
# merchant: текст + RegExp + ignorecase
flt_merchant = W.Text(placeholder="Фильтр по мерчанту (можно RegExp)")
flt_use_regex = W.Checkbox(value=True, description="RegExp")
flt_icase = W.Checkbox(value=True, description="Ignore case")

# amount: диапазон
amt_min, amt_max = float(df["amount"].min()), float(df["amount"].max())
flt_amount = W.FloatRangeSlider(value=(amt_min, amt_max), min=amt_min, max=amt_max, step=1.0,
                                description="Сумма", readout=True, layout=W.Layout(width="420px"))

# date: диапазон строкой YYYY-MM-DD (быстрый способ без парсинга дат)
date_min, date_max = str(df["date"].min()), str(df["date"].max())
flt_date_from = W.Text(value=date_min, description="Дата от")
flt_date_to   = W.Text(value=date_max, description="Дата до")

# быстрый фильтр по категории (верхний уровень)
flt_cat = W.Dropdown(options=["(все)"] + TOP_LEVEL, value="(все)", description="Категория")

btn_apply = W.Button(description="Применить фильтры", button_style="")
btn_clear = W.Button(description="Сбросить", button_style="")

# ========= 5) Таблица (шапка + контейнер строк) =========
header = W.HBox([
    W.Label("", layout=W.Layout(width="40px")),
    W.Label("Дата", layout=W.Layout(width="110px")),
    W.Label("Сумма", layout=W.Layout(width="80px")),
    W.Label("Мерчант", layout=W.Layout(width="220px")),
    W.Label("Категория", layout=W.Layout(width="220px")),
    W.Label("Подкатегория", layout=W.Layout(width="240px")),
])
header.layout = W.Layout(border="0", padding="2px 6px", margin="0 0 6px 0")
rows_box = W.VBox([])

def build_rows(mask=None):
    row_states.clear()
    rows = [header]
    indices = df.index if mask is None else df.index[mask]
    global visible_indices
    visible_indices = list(indices)
    for i in indices:
        box, st = make_row_widget(i, df.loc[i].to_dict())
        row_states.append(st)
        rows.append(box)
    rows_box.children = rows

# ========= 6) Логика фильтрации =========
def apply_filters(_=None):
    m = pd.Series(True, index=df.index)

    # merchant
    patt = flt_merchant.value.strip()
    if patt:
        if flt_use_regex.value:
            flags = re.IGNORECASE if flt_icase.value else 0
            def _match(s):
                try:
                    return bool(re.search(patt, str(s or ""), flags))
                except re.error:
                    return False
            m &= df["merchant"].apply(_match)
        else:
            s = patt.lower() if flt_icase.value else patt
            if flt_icase.value:
                m &= df["merchant"].fillna("").str.lower().str.contains(s)
            else:
                m &= df["merchant"].fillna("").str.contains(s)

    # amount
    lo, hi = flt_amount.value
    m &= (df["amount"] >= lo) & (df["amount"] <= hi)

    # date (лексикографическое сравнение строк YYYY-MM-DD)
    d_from, d_to = flt_date_from.value.strip() or date_min, flt_date_to.value.strip() or date_max
    m &= (df["date"] >= d_from) & (df["date"] <= d_to)

    # top-level category quick filter
    if flt_cat.value != "(все)":
        m &= (df["category"] == flt_cat.value)

    build_rows(mask=m)

def clear_filters(_=None):
    flt_merchant.value = ""
    flt_use_regex.value = True
    flt_icase.value = True
    flt_amount.value = (amt_min, amt_max)
    flt_date_from.value, flt_date_to.value = date_min, date_max
    flt_cat.value = "(все)"
    build_rows(mask=None)

btn_apply.on_click(apply_filters)
btn_clear.on_click(clear_filters)

# ========= 7) Сохранение и экспорт =========
def save_to_df(_=None):
    for st in row_states:
        i = st["idx"]
        cat = st["dd_cat"].value if st["dd_cat"].value != "—" else None
        sub = st["dd_sub"].value if st["dd_sub"].value != "—" else None
        df.at[i, "category"] = cat
        df.at[i, "subcategory"] = sub
    with out:
        clear_output()
        display(df)

def export_csv(_=None):
    path = "labeled_transactions.csv"
    df.to_csv(path, index=False)
    with out:
        clear_output()
        print(f"Экспортировано в {path}")

btn_save = W.Button(description="Сохранить в DataFrame", button_style="primary")
btn_export = W.Button(description="Экспорт в CSV")
btn_save.on_click(save_to_df)
btn_export.on_click(export_csv)

# ========= 8) «Протяжка» как в Excel =========
btn_filldown = W.Button(description="Протянуть из первой видимой", tooltip="Скопировать категорию/подкатегорию верхней видимой строки во все отфильтрованные")
def filldown(_=None):
    if not visible_indices:
        with out:
            clear_output()
            print("Нет видимых строк для протяжки.")
        return
    # найдём состояние первой видимой строки
    first_idx = visible_indices[0]
    st0 = next((st for st in row_states if st["idx"] == first_idx), None)
    if st0 is None:
        with out:
            clear_output()
            print("Не удалось определить первую видимую строку.")
        return
    cat_val = st0["dd_cat"].value
    sub_val = st0["dd_sub"].value

    # применяем к df и к виджетам всех видимых строк
    for st in row_states:
        i = st["idx"]
        # обновляем виджеты
        st["dd_cat"].value = cat_val
        _set_sub_options(st["dd_sub"], cat_val)
        st["dd_sub"].value = sub_val if sub_val in st["dd_sub"].options else "—"
        # обновляем df
        df.at[i, "category"] = cat_val if cat_val != "—" else None
        df.at[i, "subcategory"] = (sub_val if (sub_val != "—" and sub_val in CATEGORY_TREE.get(cat_val, []))
                                   else None)

    with out:
        clear_output()
        print(f"Протянуто: {len(visible_indices)} строк.")

btn_filldown.on_click(filldown)

# ========= 9) Сборка UI =========
filters_row1 = W.HBox([flt_merchant, flt_use_regex, flt_icase])
filters_row2 = W.HBox([flt_amount, flt_cat])
filters_row3 = W.HBox([flt_date_from, flt_date_to, btn_apply, btn_clear])
toolbar = W.HBox([btn_save, btn_export, btn_filldown])

ui = W.VBox([
    W.HTML("<b>Фильтры</b>"),
    filters_row1, filters_row2, filters_row3,
    W.HTML("<hr>"),
    rows_box,
    W.HTML("<hr>"),
    toolbar,
    out
])

build_rows(mask=None)
display(ui)